<a href="https://colab.research.google.com/github/AlemanMatias/TP-Gestion-del-mundial-Grupo-6/blob/main/python/importarArbitros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""ExportacionArbitros.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1PaNThoy9_ygBTgk3-G9xi1fch_AhqMvn
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from google.colab import files

# descargo el HTML de Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_referees"
print("Descargando la página de Wikipedia...")
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# busco todas las tablas 'wikitable'
tables = soup.find_all('table', {'class': 'wikitable'})

# La primera tabla contiene la lista histórica año por año
table = tables[0] if tables else None

def limpiar_texto(texto):
    if not texto:
        return ""
    # elimino notas al pie de Wikipedia tipo
    texto = re.sub(r'\[.*?\]', '', texto)
    # si hay muchos espacios los reduzco a uno solo
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def parse_persona(texto):
    """Extrae nombre, apellido y las siglas de 3 letras (ISO) de una cadena tipo 'John Langenus (BEL)'"""
    texto_limpio = limpiar_texto(texto)
    if not texto_limpio:
        return "", "", ""

    # extraigo las siglas del pais, estan entre parentesis
    match_pais = re.search(r'\(\s*([A-Za-z]{2,3})\s*\)', texto_limpio)
    iso = ""
    if match_pais:
        iso = match_pais.group(1).upper()
    else:
        iso = "UNK" # Unknown / Desconocido

    # quito el parentesis del nombre
    nombre_completo = re.sub(r'\s*\(\s*[A-Za-z]{2,3}\s*\)', '', texto_limpio).strip()

    partes = nombre_completo.split()
    if len(partes) >= 2:
        return partes[0], ' '.join(partes[1:]), iso
    elif len(partes) == 1:
        return partes[0], "", iso
    return "", "", ""

data = []

if table:
    rows = table.find_all('tr')
    for row in rows[1:]:
        cols = row.find_all(['td', 'th'])
        if len(cols) >= 3:
            # obtengo el arbitro principal
            ref_text = cols[1].get_text(separator=' ', strip=True)
            ref_n, ref_a, ref_p = parse_persona(ref_text)

            if not ref_n or ref_n.isdigit() or ref_n == 'Referee':
                continue

            # aca obtengo los asistentes son 2
            # uno todo el texto de la celda reemplazando saltos de línea por espacios
            asst_text_unificado = cols[2].get_text(separator=' ', strip=True)
            asst_text_unificado = re.sub(r'\s+', ' ', asst_text_unificado)

            # busco patrones tipo: Nombre Apellido (PAIS)
            # texto que terminan con un código de país entre paréntesis porque la informacion de los 2 asistentes esta en 1 sola columna
            patron_asistente = r'([^\(]+?\(\s*[A-Za-z]{2,3}\s*\))'
            asistentes_crudos = re.findall(patron_asistente, asst_text_unificado)

            asistentes_encontrados = []
            for asst_crudo in asistentes_crudos:
                n, a, p = parse_persona(asst_crudo)
                if n:
                    asistentes_encontrados.append((n, a, p))

            # asigno info del asistente 1 y asistente 2
            ast1_n, ast1_a, ast1_p = asistentes_encontrados[0] if len(asistentes_encontrados) > 0 else ("", "", "")
            ast2_n, ast2_a, ast2_p = asistentes_encontrados[1] if len(asistentes_encontrados) > 1 else ("", "", "")

            data.append({
                'arbitro_nombre': ref_n,
                'arbitro_apellido': ref_a,
                'arbitro_pais': ref_p,
                'asistente1_nombre': ast1_n,
                'asistente1_apellido': ast1_a,
                'asistente1_pais': ast1_p,
                'asistente2_nombre': ast2_n,
                'asistente2_apellido': ast2_a,
                'asistente2_pais': ast2_p,
                'categoria': 'FIFA'
            })

# creo el DataFrame y eliminar duplicados
df_arbitros = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)

# mensaje de exito
print(f"\n¡Se procesaron con éxito {len(df_arbitros)} registros con árbitros y sus respectivos asistentes!")
display(df_arbitros.head(15))

# lo guardo como archivo arbitros.csv usando codificación 'utf-8-sig' para conservar las tildes
nombre_archivo = "arbitros.csv"
df_arbitros.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f"\nArchivo '{nombre_archivo}' creado de manera local.")

# se descarga
try:
    files.download(nombre_archivo)
except Exception as e:
    print("Nota: No se pudo iniciar la descarga automática.")

Descargando la página de Wikipedia...

¡Se procesaron con éxito 22 registros con árbitros y sus respectivos asistentes!


,arbitro_nombre,arbitro_apellido,arbitro_pais,asistente1_nombre,asistente1_apellido,asistente1_pais,asistente2_nombre,asistente2_apellido,asistente2_pais,categoria
0,John,Langenus,BEL,Ulises,Saucedo,BOL,Henri,Christophe,BEL,FIFA
1,Ivan,Eklind,SWE,Louis,Baert,BEL,Mihály,Iváncsics,HUN,FIFA
2,Georges,Capdeville,FRA,Hans,Wüthrich,SUI,Augustin,Krist,BEL,FIFA
3,William,Ling,ENG,Vincenzo,Orlandini,ITA,Sandy,Griffiths,WAL,FIFA
4,Maurice,Guigue,FRA,Albert,Dusch,FRG,Juan,Gardeazábal Garay,ESP,FIFA
5,Nikolay,Latyshev,URS,Leo,Horn,NED,Bobby,Davidson,SCO,FIFA
6,Gottfried,Dienst,SUI,Tofiq,Bahramov,URS,Karol,Galba,TCH,FIFA
7,Rudi,Glöckner,GDR,Rudolf,Scheurer,SUI,Ángel,Coerezza,ARG,FIFA
8,Jack,Taylor,ENG,Alfonso,González Archundía,MEX,Ramón,Barreto Ruiz,URU,FIFA
9,Sergio,Gonella,ITA,Ramón,Barreto Ruiz,URU,Erich,Linemayr,AUT,FIFA



Archivo 'arbitros.csv' creado de manera local.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>